In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

![Image](https://powerlair.com/wp-content/uploads/2021/09/solar-panels-easily-damaged-1024x536.png)

## The above image shows how the accumulated dust on solar panels covers the panel 

## 1)**Dust detection** on solar panels is crucial for maintaining their optimal performance and preventing energy loss. 

## Solar panels rely on sunlight to generate electricity, and the accumulation of dust particles on their surface can significantly **reduce** the amount of sunlight reaching the photovoltaic cells. 

## This can lead to a decrease in electricity output, which can negatively impact energy production and cost savings.

## The key benefits of using dust detection systems on solar panels:

# Enhanced Energy Production
# Reduced Maintenance Costs
# Extended Panel Lifespan

## The **steps** followed in this project are:-

## 1)Reading images from different directories 
## 2)Re-sizing them in to one pixel range
## 3)Splitting the data in to train and test
## 4)Importing the Pre-Trained RESNET50 CNN Architecture
## 5)Pre-processing the data and defining the model
## 6)Addign the pooling, dropout and softmax layer
## 7)fitting the model and fine tuning to get better results

## In this project dataset there are **6 directories** which is having different types of solar panels images

## 1)Clean Images 

## 2)Physically damged solar panels 

## 3)Electrically damaged solar panels 

## 4)Snow covered panels

## 5)Bird dropping on solar panles

## 6)Dusty solar panels 

## The task is to predict the correct label for the given image 

## importing the required libraries for data processing and model building 

In [2]:
import tensorflow as tf
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Dropout, Activation, Flatten, Conv2D, MaxPooling2D
import numpy as np
import os
import matplotlib.pyplot as plt
# %matplotlib inline

## tf.keras.utils.image_dataset_from directory is very useful libraray expecially when working with cnn architectures which makes easy to reshape our images and splitting in to train and test dataset .

## Apart from this it also infers label from the folder names

In [3]:
image_directory = 'c:/Users/MECHREVO/OneDrive/Desktop/AI-Hybrid-Energy-Source-Predictor-main/data/solar-panel-images/Faulty_solar_panel'

# Create a training dataset from the directory
train_dataset = tf.keras.utils.image_dataset_from_directory(
    image_directory,
    image_size=(224, 224),
    batch_size=32,
    validation_split=0.2,  # Split the data into a training and validation set
    subset='training',
    shuffle=True,
    seed=42# Using 'training' to create a training dataset
)

# Create a testing (validation) dataset from the same directory
test_dataset = tf.keras.utils.image_dataset_from_directory(
    image_directory,
    image_size=(224, 224),
    batch_size=32,
    validation_split=0.2,  # Split the data into a training and validation set
    subset='validation',
    shuffle=True,
    seed=42# Using  'validation' to create a testing (validation) dataset
)

Found 300 files belonging to 6 classes.
Using 240 files for training.
Found 300 files belonging to 6 classes.
Using 60 files for validation.


In [4]:
lables=train_dataset.class_names
lables

## Plotting the images along with labels and seeing how it looks like 

In [5]:
import matplotlib.pyplot as plt

plt.figure(figsize=(18,15))

num_samples_to_display = 15
for images, labels in train_dataset.take(2):  # Only taking one batch from the training dataset
    for i in range(num_samples_to_display):
        image = images[i].numpy().astype("uint8") 
        label = labels[i]
        
        # Create a subplot
        plt.subplot(3, 5, i + 1)
        plt.imshow(image)  
        plt.title(f'Label: {lables[label]}') ## lables which is defined earlier has been used to call images labels

plt.tight_layout()  
plt.show() 

In [6]:
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import Input, Flatten, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.layers import Input, GlobalAveragePooling2D, Dense, Dropout

## using Resnet 50 pre-trained model

In [7]:

input_tensor = Input(shape=(224, 224, 3)) 
ip = tf.keras.applications.resnet50.preprocess_input(input_tensor)
base_model = ResNet50(input_tensor=ip, include_top=False, weights='imagenet')
for layer in base_model.layers:
    layer.trainable=False

In [8]:
base_model_output=base_model.output

## adding pooling and dropout layers and softmax and defining no of classes as 6 where we have to predict 6 labels 

## The number of layers and activation functions have been identied after trying out with different approaches the following architecture gives the better results

In [9]:
x = base_model.output
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = Dense(6,activation='softmax')(x)
model = tf.keras.Model(inputs=ip, outputs=outputs)

model.summary()

Model: "functional_1"
┌─────────────────────┬───────────────────┬────────────┬───────────────────┐
│ Layer (type)        │ Output Shape      │    Param # │ Connected to      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ keras_tensor_34CLO… │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_pad           │ (None, 230, 230,  │          0 │ keras_tensor_34C… │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,472 │ conv1_pad[1][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 112, 112,  │        256

In [10]:
model

In [11]:
tf.random.set_seed(42)
model.compile(optimizer=tf.keras.optimizers.Adam(0.001), loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True), metrics=['accuracy'])

# Training the model
epoch = 15
model.fit(train_dataset, validation_data=test_dataset, epochs=epoch,
    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            min_delta=1e-2,
            patience=3,
            verbose=1,
            restore_best_weights=True
        )
    ]
)
# Save the model to artifact folder
import os
os.makedirs('artifacts', exist_ok=True)
model.save('artifacts/resnet50_solar_model.h5')
print('Saved model to artifacts/resnet50_solar_model.h5')


Epoch 1/15

1/8 ━━━━━━━━━━━━━━━━━━━━ 29s 4s/step - accuracy: 0.2188 - loss: 2.0355
2/8 ━━━━━━━━━━━━━━━━━━━━ 4s 806ms/step - accuracy: 0.2031 - loss: 2.5068
3/8 ━━━━━━━━━━━━━━━━━━━━ 4s 828ms/step - accuracy: 0.2188 - loss: 2.3580
4/8 ━━━━━━━━━━━━━━━━━━━━ 3s 862ms/step - accuracy: 0.2109 - loss: 2.3098
5/8 ━━━━━━━━━━━━━━━━━━━━ 2s 859ms/step - accuracy: 0.1937 - loss: 2.2227
6/8 ━━━━━━━━━━━━━━━━━━━━ 1s 858ms/step - accuracy: 0.1875 - loss: 2.2044
7/8 ━━━━━━━━━━━━━━━━━━━━ 0s 858ms/step - accuracy: 0.1830 - loss: 2.2242
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 788ms/step - accuracy: 0.1833 - loss: 2.2047
8/8 ━━━━━━━━━━━━━━━━━━━━ 13s 1s/step - accuracy: 0.1833 - loss: 2.2047 - val_accuracy: 0.1667 - val_loss: 1.9111
Epoch 2/15

1/8 ━━━━━━━━━━━━━━━━━━━━ 5s 792ms/step - accuracy: 0.2812 - loss: 1.9701
2/8 ━━━━━━━━━━━━━━━━━━━━ 4s 824ms/step - accuracy: 0.2344 - loss: 1.9980
3/8 ━━━━━━━━━━━━━━━━━━━━ 4s 821ms/step - accuracy: 0.2188 - loss: 2.0191
4/8 ━━━━━━━━━━━━━━━━━━━━ 3s 814ms/step - accuracy: 0.2266 - lo

## Decreasing the learning rate to check how the accuracy is changing 

In [12]:
tf.random.set_seed(42)
model.compile(optimizer=tf.keras.optimizers.Adam(0.0001), loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True), metrics=['accuracy'])

epoch = 15
history = model.fit(train_dataset, validation_data=test_dataset, epochs=epoch,
    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            min_delta=1e-2,
            patience=3,
            verbose=1,
            restore_best_weights=True
        )
    ]
)
# Save the model to artifact folder
import os
os.makedirs('artifacts', exist_ok=True)
model.save('artifacts/resnet50_solar_model.h5')
print('Saved model to artifacts/resnet50_solar_model.h5')


Epoch 1/15

1/8 ━━━━━━━━━━━━━━━━━━━━ 34s 5s/step - accuracy: 0.1875 - loss: 1.8869
2/8 ━━━━━━━━━━━━━━━━━━━━ 5s 952ms/step - accuracy: 0.2188 - loss: 1.8704
3/8 ━━━━━━━━━━━━━━━━━━━━ 4s 938ms/step - accuracy: 0.2292 - loss: 1.8683
4/8 ━━━━━━━━━━━━━━━━━━━━ 3s 924ms/step - accuracy: 0.2188 - loss: 1.8527
5/8 ━━━━━━━━━━━━━━━━━━━━ 2s 911ms/step - accuracy: 0.2062 - loss: 1.8559
6/8 ━━━━━━━━━━━━━━━━━━━━ 1s 917ms/step - accuracy: 0.1979 - loss: 1.8492
7/8 ━━━━━━━━━━━━━━━━━━━━ 0s 918ms/step - accuracy: 0.1964 - loss: 1.8877
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 851ms/step - accuracy: 0.1958 - loss: 1.8867
8/8 ━━━━━━━━━━━━━━━━━━━━ 14s 1s/step - accuracy: 0.1958 - loss: 1.8867 - val_accuracy: 0.1000 - val_loss: 1.8087
Epoch 2/15

1/8 ━━━━━━━━━━━━━━━━━━━━ 6s 892ms/step - accuracy: 0.3125 - loss: 1.7803
2/8 ━━━━━━━━━━━━━━━━━━━━ 4s 803ms/step - accuracy: 0.3125 - loss: 1.8320
3/8 ━━━━━━━━━━━━━━━━━━━━ 3s 798ms/step - accuracy: 0.2604 - loss: 1.9250
4/8 ━━━━━━━━━━━━━━━━━━━━ 3s 803ms/step - accuracy: 0.2109 - lo

## 9)Hence observed better accuracy compared to higher learning rate we choose this as final prameters and proceed with test dataset for predicitons

In [13]:
test_dataset.class_names

In [14]:
training_loss = history.history['loss']
validation_loss = history.history['val_loss']
training_accuracy = history.history['accuracy'] 
validation_accuracy = history.history['val_accuracy'] 

# Create x-axis values (epochs)
epochs = range(1, len(training_loss) + 1)

# Plotting training and validation loss
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(epochs, training_loss, 'b', label='Training Loss')
plt.plot(epochs, validation_loss, 'r', label='validation Loss')
plt.title('Training and validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

# Plotting training and validation accuracy
plt.subplot(1, 2, 2)
plt.plot(epochs, training_accuracy, 'b', label='Training Accuracy')
plt.plot(epochs, validation_accuracy, 'r', label='validation Accuracy')
plt.title('Training and validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

plt.tight_layout()
plt.show()

In [15]:
# Evaluating the model on test dataset
test_loss, test_accuracy = model.evaluate(test_dataset)
print(f'Test accuracy: {test_accuracy * 100:.2f}%')


1/2 ━━━━━━━━━━━━━━━━━━━━ 0s 844ms/step - accuracy: 0.0938 - loss: 1.8059
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 715ms/step - accuracy: 0.1000 - loss: 1.8087
2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 725ms/step - accuracy: 0.1000 - loss: 1.8087
Test accuracy: 10.00%


In [16]:
loss, accuracy = model.evaluate(test_dataset)
class_names = train_dataset.class_names
plt.figure(figsize=(20, 20))
for images, labels in test_dataset.take(2):
    for i in range(32):
        ax = plt.subplot(8, 4, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        predictions = model.predict(tf.expand_dims(images[i], 0))
        score = tf.nn.softmax(predictions[0])
        if(class_names[labels[i]]==class_names[np.argmax(score)]):
            plt.title("Actual: "+class_names[labels[i]])
            plt.ylabel("Predicted: "+class_names[np.argmax(score)],fontdict={'color':'green'})
            
        else:
            plt.title("Actual: "+class_names[labels[i]])
            plt.ylabel("Predicted: "+class_names[np.argmax(score)],fontdict={'color':'red'})
        plt.gca().axes.yaxis.set_ticklabels([])        
        plt.gca().axes.xaxis.set_ticklabels([])


1/2 ━━━━━━━━━━━━━━━━━━━━ 0s 877ms/step - accuracy: 0.1250 - loss: 1.8030
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 759ms/step - accuracy: 0.1000 - loss: 1.8087
2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 769ms/step - accuracy: 0.1000 - loss: 1.8087

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step

1/1 ━━━━━━━━━

Traceback (most recent call last):
  File "C:\Users\MECHREVO\.gemini\antigravity\brain\a3ee9c8b-c0fa-44e3-8755-5708fd65ad9a\scratch\train_solar_fault_models.py", line 88, in execute_notebook
    exec(source_code, glob_ns)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 7, in <module>
  File "C:\Users\MECHREVO\OneDrive\Desktop\AI-Hybrid-Energy-Source-Predictor-main\.venv\Lib\site-packages\tensorflow\python\util\traceback_utils.py", line 167, in error_handler
    raise e.with_traceback(filtered_tb) from None
  File "C:\Users\MECHREVO\OneDrive\Desktop\AI-Hybrid-Energy-Source-Predictor-main\.venv\Lib\site-packages\tensorflow\python\framework\ops.py", line 6027, in raise_from_not_ok_status
    raise core._status_to_exception(e) from None  # pylint: disable=protected-access
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
tensorflow.python.framework.errors_impl.InvalidArgumentError: {{function_node __wrapped__StridedSlice_device_/job:localhost/replica:0/task:0/device:CPU:0}} slice ind

## 10)Hence it can be observed how our model is performing on test dataset where the red labels are shwoing means the incorrect classification given by model else the output given by model is matching with the actual label

## Hence the accuracy can be further improved by using resent variaitons of Resnet 150 and by adding some layers and parameters